### Collect zotero entries and their attachments, store the results in a file

**Warning**: misses most of what's in my Zotero DB.  
            *Maybe it's only getting entries I made since switching to zotero 7?*

In [ ]:
%load_ext autoreload
%autoreload 2

from icecream import ic
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import refwrangle.utils.refwrangle as rfw
import matplotlib.pyplot as plt

In [ ]:
def get_first_creator(item):
    """Get first creator (usually author) from zotero db top level parent item"""
    # Check if creators key exists and is not empty
    if 'creators' in item['data'] and item['data']['creators']:
        creators = item['data']['creators']
        for creator in creators:
            if creator['creatorType'] == 'author':
                if 'name' in creator:
                    return creator['name']
                else:
                    return f"{creator['lastName']}, {creator['firstName']}"
    return ''  # no creators found

def get_item_venue(item):
    """Get the venue where item appeared.  Zotero puts this in many different fields"""
    data = item['data']
    
    # Check different possible venue fields in order of priority: I know these exist
    venueKeyInPriority = [
        'publicationTitle',  # For journal articles
        'journalAbbreviation', # For journal articles
        'bookTitle',         # For book chapters
        'publisher',         # For books
        'proceedingsTitle',  # For conference papers
        'blogTitle',         # For blog posts
        'websiteTitle',      # For web pages
        'encyclopediaTitle', # For encyclopedia articles
        'dictionaryTitle',   # For dictionary entries
        'conferenceName',    # For conference papers
        'university',        # For theses
        'publisher',         # For books
        'institution',       # For reports
        'libraryCatalog',    # For library catalog entries (how zotero files YouTube)
        'place',             # For location
    ]
    
    for field in venueKeyInPriority:
        if field in data and data[field]:
            return data[field] # assume field is the venue key
        
    # If here, didn't find any of the priority venues.
    # Try to find one in this possibly ficticious dict from perplexity
    # https://www.perplexity.ai/search/for-the-item-type-forum-post-w-_4Myygs7Qni_.0iVwQ3vLg#3

    venueKeyForItemType = {
        'blogPost': 'blogTitle',
        'book': 'publisher',
        'bookSection': 'bookTitle',
        'computerProgram': 'company',
        'conferencePaper': 'proceedingsTitle',
        'dataset': 'repository',
        'dictionaryEntry': 'dictionaryTitle',
        'document': 'archive',
        'email': 'subject',
        'encyclopediaArticle': 'encyclopediaTitle',
        'forumPost': 'forumTitle',
        'journalArticle': 'publicationTitle',
        'magazineArticle': 'publicationTitle',
        'manuscript': 'archive',
        'newspaperArticle': 'publicationTitle',
        'note': 'note',
        'preprint': 'repository',
        'presentation': 'conferenceName',
        'report': 'institution',
        'thesis': 'university',
        'videoRecording': 'libraryCatalog',
        'webpage': 'websiteTitle'
    }

    try:
        itemType = data['itemType']
        return venueKeyForItemType[itemType]
    except:
        print(f"failed to find venue for {rfw.get_citation_key(data)}")
        return ''

def get_parent_metadata(parent_item, collection_names):
    """Parse a parent_item's metadata into a dict w/ standardized names in the keys"""

    # Get "author": can be many things in zotero
    pdat = parent_item['data']
    firstCreator = ''
    if 'creators' in pdat and pdat['creators']:
        creators = pdat['creators']
        ctypes = [creator['creatorType'] for creator in creators]
        hasAuthor = 'author' in ctypes # so can prioritize author creator
        for creator in creators:
            if (creator['creatorType'] == 'author') or not hasAuthor:
                if 'name' in creator:
                    firstCreator = creator['name']
                else:
                    firstCreator = f"{creator['lastName']}, {creator['firstName']}"
                break

    def get_if_there(pkey):
        return pdat[pkey] if pkey in pdat else ''

    return dict(parentFirstCreator = firstCreator,
                # convert collections from zotero keys to names
                parentCollections=[collection_names[colkey] for colkey in pdat['collections']],
                
                parentCitekey=rfw.get_citation_key(pdat),
                parentVenue=get_item_venue(parent_item),
                parentDate = get_if_there('date'),
                parentTitle = get_if_there('title'),
                parentURL=get_if_there('url'),
                parentZotkey=pdat['key'])

In [ ]:
# Read and parse the zotero database

# Remote API access
zot = zotero.Zotero(rfw.zotero_library_id, rfw.zotero_library_type, rfw.zotero_api_key)

collection_names = (defaultdict(str) # handle missed collection (W5HNMQSX for parent QTESUD23)
                    | {collection['key']: collection['data']['name'] for collection in zot.collections()})

# Get all pdf and html attachments and associate them with their parent info
# parentItems = zot.everything(zot.top())

zotero_cache = rfw.ZoteroCache()
parentItems = zotero_cache.get_data()

In [ ]:
child_exceptions, attachment_files = [], []
def process_parent_items(parentItems, collection_names):
    child_exceptions, attachment_files = [], []
    for parent in parentItems:
        if len(parent['meta']) < 1:
            continue # skip standalone notes or entries e.g. topictags.org

        pdat_always_save = get_parent_metadata(parent, collection_names)

        if rfw.is_youtube_video(parent):
            sourceInfo = pdat_always_save.copy()
            sourceInfo['contentType'] = 'youtube_video'
            attachment_files.append(sourceInfo) # not truly a file: info comes from URL
            continue

        parentCitekey = pdat_always_save['parentCitekey']
        errorParentIDstr = f'[{parentCitekey}]: {pdat_always_save["parentTitle"]}'
        for child in zot.children(parent['key']):
            if rfw.is_ignorable_child(child):
                continue

            cdat = child['data'] | pdat_always_save
            fixed = {'Fixed':False}
            try:
                cdat['file_basename'] = cdat['path'].removeprefix("attachments:")
                guessFNm = rfw.lit_attachment_dir_shared / cdat['file_basename']
                if guessFNm.exists():
                    cdat['file_fullpath'] = guessFNm
                else:
                    errStr = f'Error for {errorParentIDstr}. Full path does not exist: "{guessFNm}"'
                    print(errStr)
                    child_exceptions.append({'exception':errStr} | cdat | fixed)
            except Exception as e:
                print(f'Error for  {errorParentIDstr}: {e}')
                # No idea why these errors occur.  Try to fix
                for ext in ['pdf', 'html']: 
                    guessBasename = f'{parentCitekey}.{ext}'
                    guessFNm = rfw.lit_attachment_dir_shared / guessBasename
                    if guessFNm.exists():
                        cdat['file_basename'] = guessBasename
                        cdat['file_fullpath'] = guessFNm
                        break # if find pdf first, don't get html
                if  'file_basename' in cdat:
                    print(f"\tFix by guess basename worked: {cdat['file_basename']}")
                    fixed = {'Fixed':True}
                else:
                    print(f'\tCould not fix it. Full path does not exist: "{guessFNm}"')
                    fixed = {'Fixed':False}
                    continue # no html or pdf: don't allow it in attachment_files (below)

                child_exceptions.append({'exception': str(e)} | cdat | fixed)

            attachment_files.append(cdat)

    attachment_files = pd.DataFrame(attachment_files)
    child_exceptions = pd.DataFrame(child_exceptions)
    return child_exceptions, attachment_files

child_exceptions, attachment_files = process_parent_items(parentItems, collection_names)

attachment_files = pd.DataFrame(attachment_files)
child_exceptions = pd.DataFrame(child_exceptions)

In [ ]:
collection_key_to_name = {collection['key']:collection['data']['name'] for collection in zot.all_collections()} 
ic(collection_key_to_name);

In [ ]:
unfixed = child_exceptions.query('Fixed != True')
if (nUnfixed := len(unfixed)) > 0:
    print(f'{nUnfixed} unfixed child exceptions')
    display(unfixed)
else:
    print(f'Fixed {child_exceptions.Fixed.value_counts().values[0]} of {len(child_exceptions)} exceptions')

In [ ]:
# Find entries with unclassified venu

unclassifiedVenues = []
for parent in parentItems:
    pinfo = get_parent_metadata(parent, collection_names)
    if len(pinfo['parentVenue'])<1:
        unclassifiedVenues.append(pinfo)

if (nUnclassifVenues := len(unclassifiedVenues)) > 0:
    print(f'There were {nUnclassifVenues} unclassified venues:')
    unclassifiedVenues = pd.DataFrame(unclassifiedVenues)
    display(unclassifiedVenues)
else:
    print('No unclassified venues')

In [ ]:
# Count number of attachments (not parents) per collection
citekeysInCollection = defaultdict(list)
for row in attachment_files.itertuples(index=False):
    for collection in row.parentCollections:
        try:
            citekeysInCollection[collection].append(row.parentCitekey)
        except Exception as e:
            ic(e, row)

collection_counts = pd.Series({collection: len(filekeys) for collection, filekeys in citekeysInCollection.items()})
collection_counts.sort_values(ascending=False, inplace=True)
# plt.figure(figsize=(5, 10))  # Width is set to 8 inches, height to 5 inches
# collection_counts.plot(kind='barh',)

In [ ]:
rfw.save_pickle_data(rfw.extractedZoteroEntriesFNm, 
                 {'attachment_files':attachment_files, 
                 'child_exceptions':child_exceptions,
                 'collection_counts':collection_counts})

In [ ]:
#child_exceptions.query('Fixed != True')
unfixed.loc[1].exception

In [ ]:
collection_counts[collection_counts> 6]